### Prepare environment

In [ ]:
# =========================
# FILTERS FOR DATA EXTRACTION
# =========================
START_DATE = "20260101"
END_DATE   = "20260131"

# =========================
# VARS FOR PARALLEL PROCESSING
# =========================
MAX_WORKERS = 10


# =========================
# GLOBAL VARS TO GET FROM CONFIG FILE
# =========================

required_vars = [
    "MONGO_URI",
    "DB_NAME",
    "COLLECTION_NAME", #"simple_validations"
    "TIMEZONE",
    "OUTPUT_FILES_FOLDER",
]

In [ ]:
# from config py file

import importlib
import config

# import, cleaning the cache to get latest changes
importlib.reload(config)


print("Config file in:", config.__file__)


missing = []
locals_dict = locals()

for name in required_vars:
    if hasattr(config, name):
        locals_dict[name] = getattr(config, name)
    else:
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing config variables: {missing}")

# Create directory for output files if it doesn't exist
import os
from pathlib import Path
OUTPUT_FILES_FOLDER.mkdir(parents=True, exist_ok=True)

[name for name in required_vars if hasattr(config, name) and print(name, getattr(config, name))]

### Create conection

In [ ]:
# =========================
# Imports
# =========================
from datetime import datetime, timedelta
import pytz
from pymongo import MongoClient
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed



# =========================
# MongoDB connection
# =========================
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# =========================
# Build operational-day chunks (04:00 → 04:00)
# =========================
tz = pytz.timezone(TIMEZONE)

start_dt = tz.localize(datetime.strptime(START_DATE, "%Y%m%d")).replace(
    hour=4, minute=0, second=0, microsecond=0
)

end_dt = tz.localize(datetime.strptime(END_DATE, "%Y%m%d")).replace(
    hour=4, minute=0, second=0, microsecond=0
)

chunks = []
cursor = start_dt

while cursor <= end_dt:
    next_day = cursor + timedelta(days=1)
    chunks.append({
        "start": int(cursor.timestamp() * 1000),
        "end": int(next_day.timestamp() * 1000),
        "label": cursor.strftime("%Y%m%d")
    })
    cursor = next_day

print(f"Generated {len(chunks)} operational-day chunks")

# =========================
# Aggregation function
# =========================
def fetch_day(chunk):
    pipeline = [
        {
            "$match": {
                "created_at": {"$gte": chunk["start"], "$lt": chunk["end"]},
                "is_passenger": True
            }
        },
        {
            "$group": {
                "_id": {
                    "agency_id": "$agency_id",
                    "stop_id": "$stop_id"
                },
                "validations": {"$sum": 1}
            }
        }
    ]

    result = list(collection.aggregate(pipeline, allowDiskUse=True))

    rows = []
    for r in result:
        rows.append({
            "date": chunk["label"],
            "agency_id": r["_id"]["agency_id"],
            "stop_id": r["_id"]["stop_id"],
            "validations": r["validations"]
        })

    return rows

# =========================
# Parallel execution
# =========================
all_rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_day, c) for c in chunks]

    for i, future in enumerate(as_completed(futures), start=1):
        day_rows = future.result()
        all_rows.extend(day_rows)
        print(f"Completed {i}/{len(chunks)} chunks (+{len(day_rows)} rows)")

# =========================
# Build final DataFrame
# =========================
df_date_agency_stop_validations = pd.DataFrame(all_rows)

df_date_agency_stop_validations.sort_values(
    ["date", "agency_id", "stop_id"],
    inplace=True
)

df_date_agency_stop_validations.reset_index(drop=True, inplace=True)

#print("Final DataFrame shape:", df_date_agency_stop_validations.shape)



#df_date_agency_stop_validations.head()


In [ ]:
#save to CSV

df_date_agency_stop_validations.to_csv(os.path.join(OUTPUT_FILES_FOLDER, "date_agency_stop_validations.csv"),
     index=False
 )